# Inference: RF-DETR ONNX + SAHI — confidence threshold sweep
# Version 3. Sarah RFDETRNano model trained 1.7.1
RF-DETR Object Detection on 42 MP drone images.  
Model format: ONNX (exported from .pt checkpoint).  
Slicing: SAHI `get_sliced_prediction`, 896×896 tiles, 25% overlap.

Inference runs **once** at a low base threshold.  
mAP and confusion matrix are computed for each value in `SWEEP_THRESHOLDS` without re-running inference.

**Run order:** Cells 1–3 → **Restart session** → Cells 4 onward.

In [ ]:
!nvidia-smi

In [ ]:
# All pip installs in one place.
# After this cell completes, go to Runtime → Restart session
# (do NOT click 'Restart and run all').
# After the restart, continue running from Cell 4 onward.
!pip install -q rfdetr supervision roboflow
!pip install -q 'sahi[roboflow]'
!pip install -q 'rfdetr[onnxexport]'
!pip uninstall -y onnxruntime onnxruntime-gpu -q
!pip install -q 'onnxruntime-gpu==1.19.2'
!pip install -q 'Pillow==10.4.0'
print('Done. Now restart the runtime: Runtime → Restart session')

## ⚠️ Restart the session now

Go to **Runtime → Restart session** (do NOT click 'Restart and run all').  
After the restart, continue running from **Cell 5** downward.

In [ ]:
# Run this AFTER restarting the session to confirm the GPU provider is available.
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

import onnxruntime as ort
print('ORT version:', ort.__version__)
print('Available providers:', ort.get_available_providers())

assert 'CUDAExecutionProvider' in ort.get_available_providers(), \
    'CUDAExecutionProvider not available — check onnxruntime-gpu install'

print('PyTorch:', torch.__version__)
!pip show rfdetr

In [ ]:
## Global config — edit paths and sweep values here

# Base threshold for inference — must be <= the lowest value in SWEEP_THRESHOLDS.
# Detections below this score are discarded at SAHI inference time.
# Set to 0.00 to capture everything (SAHI still needs a small positive floor;
# 0.001 is used internally so 0.00 in the sweep is evaluated by keeping all raw preds).
SWEEP_BASE_THRESHOLD = 0.10

# Confidence thresholds to evaluate.
# Inference runs once at SWEEP_BASE_THRESHOLD; these are applied at analysis time.
SWEEP_THRESHOLDS = [0.10, 0.20, 0.25, 0.30, 0.35, 0.40,
                    0.45, 0.50, 0.60, 0.70, 0.80, 0.90, 0.95]

# IoU threshold for TP/FP/FN matching in the confusion matrix.
# Independent of confidence — 0.5 is the standard COCO value.
IOU_THRESHOLD = 0.5

SLICE_SIZE = 384   # 896 matches model training resolution

#PT_WEIGHTS        = '/content/drive/MyDrive/uavs2/checkpoint_best_total.pth'
PT_WEIGHTS        = '/content/drive/MyDrive/uavs2/sarah_checkpoint_best_total.pth'
ONNX_MODEL_PATH   = '/content/drive/MyDrive/uavs2/sara_rf_detr_nano.onnx'
ANNOTATIONS_PATH  = '/content/datasets/test/_annotations.coco.json'
TEST_IMAGES_DIR   = '/content/datasets/test/images'
OUTPUT_DIR        = '/content/datasets/output/inference'
DRIVE_OUTPUT      = '/content/drive/MyDrive/uavs2/nano_inference_results_onnx_sweep'

CSV_PATH          = f'{OUTPUT_DIR}/threshold_sweep_RF-DETR_nano_detector_extended.csv'

In [ ]:
import os
HOME = os.getcwd()
print(HOME)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

for folder in ['uavs2/', 'uavs2/output/']:
    os.makedirs('/content/drive/MyDrive/' + folder, exist_ok=True)

!mkdir -p /content/datasets/savedir/images/
!mkdir -p /content/datasets/test/images/

!rsync -a '/content/drive/MyDrive/uavs2/images/' '/content/datasets/savedir/images/'
!cp '/content/drive/MyDrive/uavs2/data.yaml' '/content/datasets/data.yaml'
!rsync -a '/content/datasets/savedir/images/' '/content/datasets/test/images/'
!cp '/content/drive/MyDrive/uavs2/_annotations.coco.json' '/content/datasets/test/_annotations.coco.json'

!ls /content/datasets/

In [ ]:
"""
test_sweep_logic.py  — run in Colab after the pip install + restart
=====================================================================
Synthetic-data tests for the threshold sweep cell.

Suites:
  A)  match_detections basic correctness
  B)  TP must be non-increasing as confidence threshold rises
  C)  mAP50 computation correctness + design diagnosis
  D)  End-to-end sweep with fully known expected values
  E)  sv.Detections boolean-mask safety (copy vs view)
"""

import numpy as np
import torch
import torchvision
from scipy.optimize import linear_sum_assignment
import supervision as sv
from supervision.metrics import MeanAveragePrecision

PASS = "\u2705 PASS"; FAIL = "\u274c FAIL"
_results = []

def check(name, actual, expected, tol=1e-4):
    ok = abs(actual - expected) <= tol if isinstance(expected, float) else actual == expected
    print(f"  {PASS if ok else FAIL}  {name}:  got={actual}  expected={expected}")
    _results.append(ok); return ok

# ── tiny helpers ─────────────────────────────────────────────────────────────

def make_gt(boxes, class_ids=None):
    xyxy = np.array(boxes, dtype=np.float32)
    cids = np.zeros(len(boxes), dtype=int) if class_ids is None else np.array(class_ids, int)
    return sv.Detections(xyxy=xyxy, class_id=cids)

def make_pred(boxes, confs, class_ids=None):
    xyxy = np.array(boxes, dtype=np.float32)
    conf = np.array(confs, dtype=np.float32)
    cids = np.zeros(len(boxes), dtype=int) if class_ids is None else np.array(class_ids, int)
    return sv.Detections(xyxy=xyxy, confidence=conf, class_id=cids)

# ── the two versions of match_detections ────────────────────────────────────

def match_detections_BUGGY(gt, pred, iou_threshold=0.5):
    """Exact copy of the current notebook cell."""
    if len(gt) == 0 and len(pred) == 0:
        return np.array([],int), np.array([],int), np.array([],int)
    elif len(gt) == 0:
        return np.array([],int), np.array([],int), np.arange(len(pred),dtype=int)
    elif len(pred) == 0:
        # BUG: returns arange(n_gt) — caller interprets len() as n_matched
        return np.arange(len(gt),dtype=int), np.array([],int), np.array([],int)
    iou = torchvision.ops.box_iou(
        torch.from_numpy(gt.xyxy), torch.from_numpy(pred.xyxy)).numpy()
    gi, pi = linear_sum_assignment(-iou)
    mg, mp, mp_set = [], [], set()
    for g, p in zip(gi, pi):
        if iou[g, p] >= iou_threshold:
            mg.append(g); mp.append(p); mp_set.add(p)
    fp = [j for j in range(len(pred)) if j not in mp_set]
    return np.array(mg,int), np.array(mp,int), np.array(fp,int)


def match_detections_FIXED(gt, pred, iou_threshold=0.5):
    """One-line fix: return empty matched_gt when no predictions exist."""
    if len(gt) == 0 and len(pred) == 0:
        return np.array([],int), np.array([],int), np.array([],int)
    elif len(gt) == 0:
        return np.array([],int), np.array([],int), np.arange(len(pred),dtype=int)
    elif len(pred) == 0:
        # FIX: no predictions → no matches.  All GT become FN via len(gt)-len(matched_gt)
        return np.array([],int), np.array([],int), np.array([],int)
    iou = torchvision.ops.box_iou(
        torch.from_numpy(gt.xyxy.astype(np.float32)),
        torch.from_numpy(pred.xyxy.astype(np.float32))).numpy()
    gi, pi = linear_sum_assignment(-iou)
    mg, mp, mp_set = [], [], set()
    for g, p in zip(gi, pi):
        if iou[g, p] >= iou_threshold:
            mg.append(g); mp.append(p); mp_set.add(p)
    fp = [j for j in range(len(pred)) if j not in mp_set]
    return np.array(mg,int), np.array(mp,int), np.array(fp,int)


def tpfpfn(fn, gt, pred, iou=0.5):
    mg, _, fp = fn(gt, pred, iou)
    return len(mg), len(fp), len(gt) - len(mg)

def filter_det(det, thresh):
    if det.confidence is None or len(det) == 0:
        return sv.Detections.empty()
    return det[det.confidence >= thresh]

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("SUITE A — match_detections_FIXED basic correctness")
print("="*62)

print("\nA1: Perfect match (exact box overlap)")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    make_gt([[100,100,200,200]]),
    make_pred([[100,100,200,200]], [0.9]))
check("TP",tp,1); check("FP",fp,0); check("FN",fn,0)

print("\nA2: No spatial overlap at all")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    make_gt([[100,100,200,200]]),
    make_pred([[900,900,999,999]], [0.9]))
check("TP",tp,0); check("FP",fp,1); check("FN",fn,1)

print("\nA3: 1 GT, 2 preds (one matches, one doesn't)")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    make_gt([[100,100,200,200]]),
    make_pred([[100,100,200,200],[900,900,999,999]], [0.9,0.8]))
check("TP",tp,1); check("FP",fp,1); check("FN",fn,0)

print("\nA4: 2 GTs, 2 perfect preds")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    make_gt([[100,100,200,200],[300,300,400,400]]),
    make_pred([[100,100,200,200],[300,300,400,400]], [0.9,0.8]))
check("TP",tp,2); check("FP",fp,0); check("FN",fn,0)

print("\nA5: 2 GTs, only 1 pred (misses second GT)")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    make_gt([[100,100,200,200],[300,300,400,400]]),
    make_pred([[100,100,200,200]], [0.9]))
check("TP",tp,1); check("FP",fp,0); check("FN",fn,1)

print("\nA6: *** BUG TRIGGER *** GT exists but predictions = EMPTY")
gt6 = make_gt([[100,100,200,200],[300,300,400,400]])
p6  = sv.Detections.empty()
print("  BUGGY version:")
tp,fp,fn = tpfpfn(match_detections_BUGGY, gt6, p6)
check("  TP should be 0  (bug returns n_gt=2)", tp, 0)
check("  FN should be 2  (bug returns 0)",      fn, 2)
print("  FIXED version:")
tp,fp,fn = tpfpfn(match_detections_FIXED, gt6, p6)
check("  TP", tp, 0); check("  FP", fp, 0); check("  FN", fn, 2)

print("\nA7: Both empty")
tp,fp,fn = tpfpfn(match_detections_FIXED, sv.Detections.empty(), sv.Detections.empty())
check("TP",tp,0); check("FP",fp,0); check("FN",fn,0)

print("\nA8: GT empty, pred exists (pure FP)")
tp,fp,fn = tpfpfn(match_detections_FIXED,
    sv.Detections.empty(), make_pred([[100,100,200,200]], [0.9]))
check("TP",tp,0); check("FP",fp,1); check("FN",fn,0)

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("SUITE B — TP must be non-increasing as threshold rises")
print("="*62)
print("""
Setup (2 images):
  img1: GT=[A]       preds=[A(conf=0.9 TP), B(conf=0.3 FP)]
  img2: GT=[C, D]    preds=[C(conf=0.5 TP)]
  ← at thresh > 0.5, img2 has zero predictions → bug fires
""")
gt1 = make_gt([[100,100,200,200]])
r1  = make_pred([[100,100,200,200],[300,300,400,400]], [0.9, 0.3])
gt2 = make_gt([[500,500,600,600],[700,700,800,800]])
r2  = make_pred([[500,500,600,600]], [0.5])
targets_b = [gt1, gt2];  raw_b = [r1, r2]
thresholds_b = [0.0, 0.35, 0.55, 0.95]

expected_b = {0.00:(2,1,1), 0.35:(2,0,1), 0.55:(1,0,2), 0.95:(0,0,3)}

print(f"  {'thresh':>6}  {'exp TP,FP,FN':>14}  {'BUGGY':>18}  {'FIXED':>18}")
tp_buggy=[]; tp_fixed=[]
for thresh in thresholds_b:
    fp_list = [filter_det(d, thresh) for d in raw_b]
    tb=fb=nb=tf=ff=nf=0
    for g,p in zip(targets_b, fp_list):
        t,f,n=tpfpfn(match_detections_BUGGY,g,p); tb+=t;fb+=f;nb+=n
        t,f,n=tpfpfn(match_detections_FIXED, g,p); tf+=t;ff+=f;nf+=n
    tp_buggy.append(tb); tp_fixed.append(tf)
    e=expected_b[thresh]
    prev_b = tp_buggy[-2] if len(tp_buggy)>1 else tb
    prev_f = tp_fixed[-2] if len(tp_fixed)>1 else tf
    b_flag = " \u2b06WRONG" if tb > prev_b else " ok"
    f_flag = " \u2b06WRONG" if tf > prev_f else " ok"
    print(f"  {thresh:>6.2f}  {str(e):>14}  {tb},{fb},{nb}{b_flag:>8}  {tf},{ff},{nf}{f_flag:>8}")
    _results.append(tf == e[0] and ff == e[1] and nf == e[2])

mono_b = all(tp_buggy[i]>=tp_buggy[i+1] for i in range(len(tp_buggy)-1))
mono_f = all(tp_fixed[i]>=tp_fixed[i+1] for i in range(len(tp_fixed)-1))
print()
check("TP non-increasing BUGGY (should be False)", mono_b, False)
check("TP non-increasing FIXED (should be True)",  mono_f, True)

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("SUITE C — mAP50 correctness and design diagnosis")
print("="*62)

# C1: perfect detector → mAP ≈ 1.0
gt_c1 = [make_gt([[100,100,200,200]])]
pr_c1 = [make_pred([[100,100,200,200]], [0.9])]
m1    = MeanAveragePrecision().update(pr_c1, gt_c1).compute()
print(f"\nC1: Perfect match  mAP50={m1.map50:.4f}  (expected ≈ 1.0)")
check("mAP50 perfect match ≈ 1.0", m1.map50, 1.0, tol=0.02)

# C2: no predictions → mAP = 0
gt_c2 = [make_gt([[100,100,200,200]])]
pr_c2 = [sv.Detections.empty()]
m2    = MeanAveragePrecision().update(pr_c2, gt_c2).compute()
print(f"\nC2: No predictions  mAP50={m2.map50:.4f}  (expected 0.0)")
check("mAP50 no predictions = 0", m2.map50, 0.0, tol=0.001)

# C3: class_id MISMATCH — diagnoses near-zero mAP in real data
# If this returns 0.0, it means class_id mismatch silently kills mAP
gt_c3 = [make_gt([[100,100,200,200]], class_ids=[0])]
pr_c3 = [make_pred([[100,100,200,200]], [0.9], class_ids=[1])]
m3    = MeanAveragePrecision().update(pr_c3, gt_c3).compute()
print(f"\nC3: class_id mismatch (GT=0, pred=1)  mAP50={m3.map50:.4f}  (expected 0.0)")
check("mAP50 class mismatch = 0 (this is how supervision silently hides mismatches)", m3.map50, 0.0, tol=0.001)

# C4: compute mAP on full vs pre-filtered — shows why pre-filtering distorts mAP
print("\nC4: mAP on full raw preds vs pre-filtered (design issue illustration)")
gt_c4   = [make_gt([[100,100,200,200]])]
raw_c4  = [make_pred([[100,100,200,200],[900,900,999,999]], [0.8, 0.3])]
filt_c4 = [filter_det(raw_c4[0], 0.5)]  # only TP survives
m_full  = MeanAveragePrecision().update(raw_c4,  gt_c4).compute()
m_filt  = MeanAveragePrecision().update(filt_c4, gt_c4).compute()
print(f"  mAP50 on full raw preds:            {m_full.map50:.4f}")
print(f"  mAP50 on pre-filtered (thresh=0.5): {m_filt.map50:.4f}")
print(f"  NOTE: these differ because mAP is an AUC property, not a point metric.")
print(f"  DESIGN FIX: compute mAP ONCE on all_raw_preds before any filtering.")

# C5: diagnose the actual mAP values from the CSV
print("""
C5: Diagnosing mAP50 ≈ 0.008 from the CSV
  At thresh=0.95:  precision=0.687, recall=0.372  (from our TP/FP/FN counts)
  Expected mAP50 for a curve with P=0.69 @ R=0.37:  ≈ 15–25%
  Actual mAP50 = 0.004  ← 40–60× lower than expected

  Most likely causes (run these diagnostic cells in order):
  1. Print ds.classes and GT class_id sample:
       path, img, ann = ds[0]
       print('GT class_ids:', ann.class_id[:5])  # should be [0, 0, ...]
       print('Classes:', ds.classes)

  2. Print prediction class_id sample:
       result = get_sliced_prediction(img, detection_model, ...)
       dets = sahi_result_to_sv_detections(result)
       print('Pred class_ids:', dets.class_id[:5])  # hardcoded 0

  3. Run a one-image mAP sanity check:
       from supervision.metrics import MeanAveragePrecision
       path, img_arr, ann = ds[0]
       img = PIL.Image.open(path)
       result = get_sliced_prediction(img, detection_model, ...)
       dets = sahi_result_to_sv_detections(result)
       m = MeanAveragePrecision().update([dets], [ann]).compute()
       print('Single-image mAP50:', m.map50)
       # If this is near 0 but you can visually see correct detections,
       # class_id mismatch is the culprit.
""")

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("SUITE D — Full sweep simulation with known expected values")
print("="*62)
print("""
3 images:
  img0: GT=[A]         preds=[A(0.9,TP), X(0.4,FP)]
  img1: GT=[B, C]      preds=[B(0.6,TP), Y(0.2,FP)]  at thresh>0.6 → 0 preds
  img2: GT=[D, E, F]   preds=[] at ALL thresholds      ← bug always fires here
""")
t_d = [make_gt([[10,10,20,20]]),
       make_gt([[30,30,40,40],[50,50,60,60]]),
       make_gt([[70,70,80,80],[90,90,100,100],[110,110,120,120]])]
r_d = [make_pred([[10,10,20,20],[200,200,210,210]], [0.9,0.4]),
       make_pred([[30,30,40,40],[300,300,310,310]], [0.6,0.2]),
       sv.Detections.empty()]

#   thresh | BUGGY (TP,FP,FN)  | FIXED (TP,FP,FN)
#   0.00   | (5, 2, 1)         | (2, 2, 4)    img2: BUGGY TP+=3, FIXED TP+=0/FN+=3
#   0.50   | (5, 0, 1)         | (2, 0, 4)    B(0.6)>=0.5 survives; img2 same
#   0.70   | (6, 0, 0)         | (1, 0, 5)    img1 also 0 preds; img2 same
#   0.95   | (6, 0, 0)         | (0, 0, 6)    all 0 preds
exp = {
    0.00: {"buggy":(5,2,1), "fixed":(2,2,4)},
    0.50: {"buggy":(5,0,1), "fixed":(2,0,4)},
    0.70: {"buggy":(6,0,0), "fixed":(1,0,5)},
    0.95: {"buggy":(6,0,0), "fixed":(0,0,6)},
}
for thresh, e in exp.items():
    fp_list = [filter_det(d, thresh) for d in r_d]
    tb=fb=nb=tf=ff=nf=0
    for g,p in zip(t_d, fp_list):
        t,f,n=tpfpfn(match_detections_BUGGY,g,p); tb+=t;fb+=f;nb+=n
        t,f,n=tpfpfn(match_detections_FIXED, g,p); tf+=t;ff+=f;nf+=n
    ok_b=(tb,fb,nb)==e["buggy"]; ok_f=(tf,ff,nf)==e["fixed"]
    print(f"thresh={thresh:.2f}")
    print(f"  BUGGY {tb},{fb},{nb}  {'✅' if ok_b else '❌'}  expected {e['buggy']}")
    print(f"  FIXED {tf},{ff},{nf}  {'✅' if ok_f else '❌'}  expected {e['fixed']}")
    _results.append(ok_b); _results.append(ok_f)

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
print("SUITE E — sv.Detections boolean mask returns independent copy")
print("="*62)
orig = make_pred([[100,100,200,200],[300,300,400,400]], [0.9,0.3])
filt = orig[orig.confidence >= 0.5]
print(f"\nOriginal length before mask: {len(orig)}")
print(f"Filtered length (conf≥0.5):  {len(filt)}")
print(f"Original length after mask:  {len(orig)}")
check("Original unchanged after mask",    len(orig), 2)
check("Filtered has correct length",      len(filt), 1)
check("Filtered confidence value",        float(filt.confidence[0]), 0.9, tol=1e-5)

# ════════════════════════════════════════════════════════════════════════
print("\n" + "="*62)
n_pass = sum(_results); n_fail = len(_results)-n_pass
print(f"TOTAL:  {n_pass} pass,  {n_fail} fail  (of {len(_results)} checks)")
print("Expected: the 2 BUGGY checks in A6 will fail — that demonstrates the bug.")
print("All FIXED checks should pass.")
print("="*62)

In [ ]:
import gc, torch, weakref

def cleanup_gpu_memory(obj=None, verbose: bool = False):
    if not torch.cuda.is_available():
        if verbose:
            print('[INFO] CUDA not available. No GPU cleanup needed.')
        return
    def get_memory_stats():
        return torch.cuda.memory_allocated(), torch.cuda.memory_reserved()
    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = get_memory_stats()
        print(f'[Before] Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB')
    if obj is not None:
        ref = weakref.ref(obj)
        del obj
        if ref() is not None and verbose:
            print('[WARNING] Object not fully garbage collected yet.')
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    torch.cuda.synchronize()
    if verbose:
        alloc, reserv = get_memory_stats()
        print(f'[After]  Allocated: {alloc / 1024**2:.2f} MB | Reserved: {reserv / 1024**2:.2f} MB')

In [ ]:
# Utility: convert YOLO format labels → COCO JSON.
# Not needed if _annotations.coco.json already exists on Drive.
import os, json
from PIL import Image

def convert_yolo_to_coco(yolo_images_path, yolo_labels_path, output_coco_path, class_names):
    coco_data = {
        'info': {'year': '2025', 'version': '0', 'description': 'UAVS',
                 'contributor': '', 'url': '', 'date_created': '2025-01-01T00:00:00+00:00'},
        'licenses': [{'id': 1, 'url': 'https://creativecommons.org/licenses/by/4.0/', 'name': 'CC BY 4.0'}],
        'images': [], 'annotations': [], 'categories': []
    }
    for i, name in enumerate(class_names):
        coco_data['categories'].append({'id': i, 'name': name, 'supercategory': 'none'})
    image_id = annotation_id = 0
    for img_filename in os.listdir(yolo_images_path):
        if not img_filename.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path   = os.path.join(yolo_images_path, img_filename)
        label_path = os.path.join(yolo_labels_path, os.path.splitext(img_filename)[0] + '.txt')
        with Image.open(img_path) as img:
            width, height = img.size
        coco_data['images'].append({'id': image_id, 'file_name': img_filename,
                                    'width': width, 'height': height})
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    cid, xc, yc, bw, bh = map(float, parts)
                    x_min = (xc - bw / 2) * width
                    y_min = (yc - bh / 2) * height
                    abs_bw, abs_bh = bw * width, bh * height
                    coco_data['annotations'].append({
                        'id': annotation_id, 'image_id': image_id,
                        'category_id': int(cid),
                        'bbox': [x_min, y_min, abs_bw, abs_bh],
                        'area': abs_bw * abs_bh, 'iscrowd': 0
                    })
                    annotation_id += 1
        image_id += 1
    with open(output_coco_path, 'w') as f:
        json.dump(coco_data, f, indent=4)

In [ ]:
## Export .pt checkpoint → ONNX.
## Skipped automatically if the .onnx already exists on Drive.
import os, json, shutil
from rfdetr import RFDETRNano

with open(ANNOTATIONS_PATH) as f:
    coco_meta = json.load(f)

CLASS_NAMES      = [cat['name'] for cat in sorted(coco_meta['categories'], key=lambda c: c['id'])]
CATEGORY_MAPPING = {str(i): name for i, name in enumerate(CLASS_NAMES)}
print(f'Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}')
print(f'CATEGORY_MAPPING: {CATEGORY_MAPPING}')

if os.path.exists(ONNX_MODEL_PATH):
    print(f'\nONNX already exists, skipping export:\n  {ONNX_MODEL_PATH}')
else:
    print(f'\nExporting {PT_WEIGHTS} → ONNX ...')
    export_model = RFDETRNano(
        pretrain_weights=PT_WEIGHTS,
        resolution=384,
        num_classes=len(CLASS_NAMES),
    )
    export_model.export()
    shutil.copy('output/rfdetr-nano.onnx', ONNX_MODEL_PATH)
    del export_model
    cleanup_gpu_memory(verbose=True)
    print(f'Exported and saved to {ONNX_MODEL_PATH}')

In [ ]:
## Load the ONNX model into a custom SAHI DetectionModel subclass.
## detection_model and sahi_result_to_sv_detections() are used by all cells below.

import torch, numpy as np, onnxruntime as ort, supervision as sv
from PIL import Image
from sahi.models.base import DetectionModel
from sahi.prediction import ObjectPrediction
from sahi.predict import get_sliced_prediction


class RFDetrOnnxDetectionModel(DetectionModel):
    """SAHI wrapper for RF-DETR ONNX.
    ONNX outputs (rfdetr 1.5.1):
      dets:   [1, 300, 4]  normalised cx,cy,w,h
      labels: [1, 300, 1]  raw logits  (sigmoid → confidence)
    """

    def load_model(self):
        providers = (
            ['CUDAExecutionProvider', 'CPUExecutionProvider']
            if torch.cuda.is_available() else ['CPUExecutionProvider']
        )
        self.session      = ort.InferenceSession(self.model_path, providers=providers)
        self.input_name   = self.session.get_inputs()[0].name
        self.output_names = [o.name for o in self.session.get_outputs()]
        print(f'ONNX inputs : {self.input_name}')
        print(f'ONNX outputs: {self.output_names}')
        print(f'Active provider: {self.session.get_providers()[0]}')
        self.model = self.session

    def perform_inference(self, image: np.ndarray):
        h, w  = image.shape[:2]
        tile  = Image.fromarray(image).resize((self.image_size, self.image_size), Image.BILINEAR)
        tile  = np.array(tile, dtype=np.float32) / 255.0
        mean  = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std   = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        tile  = ((tile - mean) / std).transpose(2, 0, 1)[None]
        self._original_predictions = self.session.run(self.output_names, {self.input_name: tile})
        self._tile_hw = (h, w)

    def convert_original_predictions(self, shift_amount=None, full_shape=None):
      if shift_amount is None:
          shift_amount = [0, 0]

      dets   = self._original_predictions[0][0]   # [300, 4]  cx,cy,w,h normalised
      labels = self._original_predictions[1][0]   # [300, 1]  raw logits
      tile_h, tile_w = self._tile_hw

      # Confidence = sigmoid(logit)
      scores = 1.0 / (1.0 + np.exp(-labels[:, 0]))   # [300]

      object_prediction_list = []
      for box, score in zip(dets, scores):
          if score < self.confidence_threshold:
              continue
          cx, cy, bw, bh = box
          x1 = (cx - bw / 2) * tile_w
          y1 = (cy - bh / 2) * tile_h
          x2 = (cx + bw / 2) * tile_w
          y2 = (cy + bh / 2) * tile_h

          # Clip to tile boundaries
          x1, y1 = max(0.0, x1), max(0.0, y1)
          x2, y2 = min(float(tile_w), x2), min(float(tile_h), y2)

          if x2 <= x1 or y2 <= y1:
              continue

          # Apply tile offset to convert tile-local → full image coordinates
          x1 += shift_amount[0]
          y1 += shift_amount[1]
          x2 += shift_amount[0]
          y2 += shift_amount[1]

          category_name = self.category_mapping.get("0", "object")
          object_prediction_list.append(
              ObjectPrediction(
                  bbox=[x1, y1, x2, y2],
                  score=float(score),
                  category_id=0,
                  category_name=category_name,
                  shift_amount=[0, 0],   # already applied above
                  full_shape=full_shape,
              )
          )
      self._object_prediction_list_per_image = [object_prediction_list]


def sahi_result_to_sv_detections(result) -> sv.Detections:
    """Convert SAHI PredictionResult → supervision.Detections."""
    preds = result.object_prediction_list
    if not preds:
        return sv.Detections.empty()
    xyxy       = np.array([[p.bbox.minx, p.bbox.miny, p.bbox.maxx, p.bbox.maxy]
                            for p in preds], dtype=np.float32)
    confidence = np.array([p.score.value for p in preds], dtype=np.float32)
    class_ids  = np.array([p.category.id  for p in preds], dtype=int)
    return sv.Detections(xyxy=xyxy, confidence=confidence, class_id=class_ids)


DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f'Loading ONNX model on {DEVICE} at base threshold {SWEEP_BASE_THRESHOLD} ...')
detection_model = RFDetrOnnxDetectionModel(
    model_path=ONNX_MODEL_PATH,
    confidence_threshold=SWEEP_BASE_THRESHOLD,   # low — captures all candidates
    category_mapping=CATEGORY_MAPPING,
    image_size=384,
    device=DEVICE,
)
detection_model.load_model()
print('Model loaded.\n')

In [ ]:
import supervision as sv

ds = sv.DetectionDataset.from_coco(
    images_directory_path=TEST_IMAGES_DIR,
    annotations_path=ANNOTATIONS_PATH,
)
print(f'Dataset loaded: {len(ds)} images, classes: {ds.classes}')

In [ ]:
## Run inference ONCE at SWEEP_BASE_THRESHOLD. Cell 13
## Raw detections are saved; the sweep below re-filters them without re-running inference.
from tqdm import tqdm
from PIL import Image

targets       = []
all_raw_preds = []   # detections at base threshold, one sv.Detections per image

for path, image, annotations in tqdm(ds, desc=f'Inference (base threshold={SWEEP_BASE_THRESHOLD})'):
    image_pil  = Image.open(path)
    result     = get_sliced_prediction(
        image_pil, detection_model,
        slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
        overlap_height_ratio=0.25, overlap_width_ratio=0.25,
        verbose=0,
    )
    detections = sahi_result_to_sv_detections(result)
    if detections.class_id is not None:
        detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)
    targets.append(annotations)
    all_raw_preds.append(detections)

print(f'\nInference complete. Raw detections per image (mean): '
      f'{sum(len(d) for d in all_raw_preds) / len(all_raw_preds):.1f}')

In [ ]:
## DIAGNOSTIC 1: verify GT class IDs from supervision dataset
path, img_arr, ann = ds[0]
print('Classes in dataset:', ds.classes)
print('GT class_ids (first image, first 10):', ann.class_id[:10])
print('Expected: all zeros for a single-class dataset')
print()
# If you see class_id=1 here, that means supervision loaded COCO 1-indexed
# IDs and did NOT remap them. Your predictions hardcode category_id=0,
# so mAP would be computed as class 0 vs class 1 → zero matches → mAP=0.

In [ ]:
## DIAGNOSTIC 2: verify prediction class IDs from the ONNX+SAHI pipeline
import PIL.Image
_path, _img_arr, _ann = ds[0]
_img = PIL.Image.open(_path)
_result = get_sliced_prediction(
    _img, detection_model,
    slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
    overlap_height_ratio=0.25, overlap_width_ratio=0.25,
    verbose=0,
)
_dets = sahi_result_to_sv_detections(_result)
print(f'Detections found: {len(_dets)}')
if len(_dets) > 0:
    print('Pred class_ids (first 10):', _dets.class_id[:10])
    print('Pred confidence (first 5): ', _dets.confidence[:5])
else:
    print('No detections at base threshold — try lowering SWEEP_BASE_THRESHOLD')
print()
print('GT  class_ids (first 10):', _ann.class_id[:10])
print()
print('These must match for mAP to work.')
print('If GT=1 and Pred=0, that is the mismatch.')

In [ ]:
## DIAGNOSTIC 3: single-image mAP sanity check
## If this returns near 0 despite visible correct detections, class_id mismatch confirmed.
from supervision.metrics import MeanAveragePrecision

_m = MeanAveragePrecision().update([_dets], [_ann]).compute()
print(f'Single-image mAP50:    {_m.map50:.4f}')
print(f'Single-image mAP50:95: {_m.map50_95:.4f}')
print()
if _m.map50 < 0.01 and len(_dets) > 0:
    print('⚠️  mAP near zero despite detections existing.')
    print('   Almost certainly a class_id mismatch.')
    print('   Fix: change category_id=0 in convert_original_predictions')
    print('   to match whatever class_id is in ann.class_id above.')
elif _m.map50 > 0.1:
    print('✅ mAP looks reasonable. The sweep mAP issue is the pre-filtering design.')
    print('   Move MeanAveragePrecision computation outside the sweep loop.')

In [ ]:
## Standalone mAP diagnosis — run as a new cell
import PIL.Image
import numpy as np
from supervision.metrics import MeanAveragePrecision

path, img_arr, ann = ds[0]
img = PIL.Image.open(path)

result = get_sliced_prediction(
    img, detection_model,
    slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
    overlap_height_ratio=0.25, overlap_width_ratio=0.25,
    verbose=0,
)

preds = result.object_prediction_list
print(f'Raw predictions from SAHI: {len(preds)}')
if preds:
    print(f'First pred: bbox={preds[0].bbox.to_xyxy()}, score={preds[0].score.value:.4f}, category_id={preds[0].category.id}')

dets = sahi_result_to_sv_detections(result)
print(f'\nsv.Detections: {len(dets)}')
print(f'xyxy[0]:       {dets.xyxy[0]}')
print(f'confidence[0]: {dets.confidence[0]:.4f}')
print(f'class_id[0]:   {dets.class_id[0]}')

print(f'\nGT boxes:      {len(ann)}')
print(f'GT xyxy[0]:    {ann.xyxy[0]}')
print(f'GT class_id[0]:{ann.class_id[0]}')

print(f'\nxyxy dtype pred: {dets.xyxy.dtype}')
print(f'xyxy dtype GT:   {ann.xyxy.dtype}')

import torchvision, torch
iou = torchvision.ops.box_iou(
    torch.from_numpy(ann.xyxy[:5].astype(np.float32)),
    torch.from_numpy(dets.xyxy[:5].astype(np.float32))
)
print(f'\nIoU matrix (first 5 GT vs first 5 preds):\n{iou}')

m = MeanAveragePrecision().update([dets], [ann]).compute()
print(f'\nmAP50: {m.map50:.4f}')

In [ ]:
from sahi.prediction import ObjectPrediction
op = ObjectPrediction(
    bbox=[100, 100, 200, 200],
    score=0.9,
    category_id=0,
    category_name="object",
    shift_amount=[500, 300],
    full_shape=None,
)
print("minx:", op.bbox.minx)  # should be 600 if shift is applied
print("miny:", op.bbox.miny)  # should be 400 if shift is applied

In [ ]:
## Sweep over SWEEP_THRESHOLDS.
## For each threshold: filter raw detections, compute TP/FP/FN,
## plot and save a confusion matrix, then write a summary CSV.
## mAP50 is computed ONCE on all_raw_preds before the loop.

import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision, os, shutil, csv
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import confusion_matrix
from supervision.metrics import MeanAveragePrecision


def match_detections(ground_truth, predictions, iou_threshold=0.5):
    if len(ground_truth) == 0 and len(predictions) == 0:
        return np.array([], int), np.array([], int), np.array([], int)
    elif len(ground_truth) == 0:
        return np.array([], int), np.array([], int), np.arange(len(predictions), dtype=int)
    elif len(predictions) == 0:
        return np.array([], int), np.array([], int), np.array([], int)

    iou_matrix = torchvision.ops.box_iou(
        torch.from_numpy(ground_truth.xyxy),
        torch.from_numpy(predictions.xyxy)
    ).numpy()
    gt_idx_arr, pred_idx_arr = linear_sum_assignment(-iou_matrix)

    matched_gt, matched_pred, matched_pred_set = [], [], set()
    for g, p in zip(gt_idx_arr, pred_idx_arr):
        if iou_matrix[g, p] >= iou_threshold:
            matched_gt.append(g)
            matched_pred.append(p)
            matched_pred_set.add(p)

    fp = [j for j in range(len(predictions)) if j not in matched_pred_set]
    return np.array(matched_gt, int), np.array(matched_pred, int), np.array(fp, int)


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

# Compute mAP ONCE on all raw predictions before any threshold filtering
map_result = MeanAveragePrecision().update(all_raw_preds, targets).compute()
MAP50 = map_result.map50
print(f'mAP@50:    {MAP50:.4f}')
print(f'mAP@50:95: {map_result.map50_95:.4f}')

csv_rows = []

for thresh in SWEEP_THRESHOLDS:
    print(f'\n{"="*62}')
    print(f'  Confidence threshold = {thresh:.2f}')
    print(f'{"="*62}')

    # Filter raw detections at this threshold
    filtered_preds = []
    for det in all_raw_preds:
        if det.confidence is None or len(det) == 0:
            filtered_preds.append(sv.Detections.empty())
        else:
            mask = det.confidence >= thresh
            filtered_preds.append(det[mask])

    n_total = sum(len(d) for d in filtered_preds)
    print(f'  Detections surviving filter: {n_total}  '
          f'(mean {n_total/max(len(filtered_preds),1):.1f}/image)')

    # TP / FP / FN
    total_tp = total_fp = total_fn = 0
    true_labels = []
    predicted_labels = []

    for gt_det, pred_det in zip(targets, filtered_preds):
        if gt_det is None or pred_det is None:
            continue
        matched_gt, matched_pred, fp_idx = match_detections(
            gt_det, pred_det, IOU_THRESHOLD
        )
        tp = len(matched_gt)
        fp = len(fp_idx)
        fn = len(gt_det) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn

        for g, p in zip(matched_gt, matched_pred):
            true_labels.append(gt_det.class_id[g])
            predicted_labels.append(pred_det.class_id[p])
        for p in fp_idx:
            true_labels.append(-1)
            predicted_labels.append(pred_det.class_id[p])
        for g in list(set(range(len(gt_det))) - set(matched_gt.tolist())):
            true_labels.append(gt_det.class_id[g])
            predicted_labels.append(-1)

    precision   = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0.0
    recall      = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0.0
    f1          = (2 * precision * recall / (precision + recall)
                   if (precision + recall) > 0 else 0.0)
    fp_tp_ratio = total_fp / total_tp if total_tp > 0 else float('inf')

    print(f'  TP={total_tp}  FP={total_fp}  FN={total_fn}')
    print(f'  Precision={precision:.4f}  Recall={recall:.4f}  '
          f'F1={f1:.4f}  FP/TP={fp_tp_ratio:.4f}')

    csv_rows.append({
        'Threshold':   thresh,
        'mAP50':       round(MAP50, 6),
        'FP':          total_fp,
        'TP':          total_tp,
        'FN':          total_fn,
        'Precision':   round(precision, 6),
        'Recall':      round(recall, 6),
        'F1':          round(f1, 6),
        'FP_TP_Ratio': round(fp_tp_ratio, 6) if fp_tp_ratio != float('inf') else 'inf',
    })

    # Confusion matrix
    all_cls       = sorted([l for l in set(true_labels + predicted_labels) if l != -1])
    unique_labels = all_cls + ([-1] if -1 in set(true_labels + predicted_labels) else [])
    label_to_idx  = {label: i for i, label in enumerate(unique_labels)}
    true_idx      = [label_to_idx[l] for l in true_labels]
    pred_idx_list = [label_to_idx[l] for l in predicted_labels]
    cm            = confusion_matrix(true_idx, pred_idx_list,
                                     labels=list(label_to_idx.values()))

    cm_labels = [None] * len(unique_labels)
    for label, idx in label_to_idx.items():
        if label == -1:
            cm_labels[idx] = 'No Object'
        else:
            try:
                cm_labels[idx] = ds.classes[label]
            except IndexError:
                cm_labels[idx] = f'Class {label}'

    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]), yticks=np.arange(cm.shape[0]),
        xticklabels=cm_labels, yticklabels=cm_labels,
        title=(f'conf={thresh:.2f}  IoU={IOU_THRESHOLD}  '
               f'mAP@50={MAP50:.3f}  P={precision:.3f}  R={recall:.3f}  F1={f1:.3f}'),
        ylabel='True label', xlabel='Predicted label'
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    fmt     = '.2f' if cm.dtype.kind == 'f' else 'd'
    thresh_v = cm.max() / 2. if cm.max() > 0 else 1
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt), ha='center', va='center',
                    color='white' if cm[i, j] > thresh_v else 'black')
    fig.tight_layout()

    conf_str   = f'{thresh:.2f}'.replace('.', '_')
    local_path = f'{OUTPUT_DIR}/confusion_matrix_conf_{conf_str}.png'
    drive_path = f'{DRIVE_OUTPUT}/confusion_matrix_conf_{conf_str}.png'
    plt.savefig(local_path)
    plt.show()
    try:
        shutil.copy2(local_path, drive_path)
    except Exception as e:
        print(f'  Could not copy confusion matrix to Drive: {e}')

# Write CSV
csv_columns = ['Threshold', 'mAP50', 'FP', 'TP', 'FN',
               'Precision', 'Recall', 'F1', 'FP_TP_Ratio']
with open(CSV_PATH, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=csv_columns)
    writer.writeheader()
    writer.writerows(csv_rows)

drive_csv = f'{DRIVE_OUTPUT}/threshold_sweep_RF-DETR_large_detector_extended.csv'
try:
    shutil.copy2(CSV_PATH, drive_csv)
    print(f'\nCSV saved to Drive: {drive_csv}')
except Exception as e:
    print(f'\nCSV saved locally only ({CSV_PATH}): {e}')

# Print summary table
header = (f'  {"Thresh":>7}  {"mAP50":>7}  {"TP":>7}  {"FP":>7}  '
          f'{"FN":>7}  {"Prec":>7}  {"Rec":>7}  {"F1":>7}  {"FP/TP":>8}')
print(f'\n{"-" * len(header)}')
print(header)
print(f'{"-" * len(header)}')
for row in csv_rows:
    fpratio = f"{row['FP_TP_Ratio']:.4f}" if row['FP_TP_Ratio'] != 'inf' else '     inf'
    print(f"  {row['Threshold']:>7.2f}  {row['mAP50']:>7.4f}  {row['TP']:>7d}  "
          f"{row['FP']:>7d}  {row['FN']:>7d}  {row['Precision']:>7.4f}  "
          f"{row['Recall']:>7.4f}  {row['F1']:>7.4f}  {fpratio:>8}")
print(f'{"-" * len(header)}')

In [ ]:
## Single-image visualisation at a chosen threshold.
## Change VISUALISE_THRESHOLD and IMAGE_INDEX as needed.
VISUALISE_THRESHOLD = 0.45
IMAGE_INDEX         = 51

from PIL import Image

path, image, annotations = ds[IMAGE_INDEX]
image = Image.open(path)

result = get_sliced_prediction(
    image, detection_model,
    slice_height=SLICE_SIZE, slice_width=SLICE_SIZE,
    overlap_height_ratio=0.5, overlap_width_ratio=0.5,
    verbose=0,
)
detections = sahi_result_to_sv_detections(result)
if detections.confidence is not None and len(detections) > 0:
    detections = detections[detections.confidence >= VISUALISE_THRESHOLD]
if detections.class_id is not None:
    detections.class_id = np.clip(detections.class_id, 0, len(CLASS_NAMES) - 1)

text_scale = sv.calculate_optimal_text_scale(resolution_wh=image.size)
thickness  = sv.calculate_optimal_line_thickness(resolution_wh=image.size)
color = sv.ColorPalette.from_hex([
    '#ffff00', '#ff9b00', '#ff66ff', '#3399ff', '#ff66b2', '#ff8080',
    '#b266ff', '#9999ff', '#66ffff', '#33ff99', '#66ff66', '#99ff00'
])
bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK,
                                     text_scale=text_scale)

annotations_labels = [f'{ds.classes[cid]}' for cid in annotations.class_id]
detections_labels  = [f'{ds.classes[cid]} {conf:.2f}'
                      for cid, conf in zip(detections.class_id, detections.confidence)]

ann_img = bbox_annotator.annotate(image.copy(), annotations)
ann_img = label_annotator.annotate(ann_img, annotations, annotations_labels)
det_img = bbox_annotator.annotate(image.copy(), detections)
det_img = label_annotator.annotate(det_img, detections, detections_labels)

sv.plot_images_grid(images=[ann_img, det_img], grid_size=(1, 2),
                   titles=['Ground Truth', f'ONNX + SAHI  conf>={VISUALISE_THRESHOLD}'])

In [ ]:
## Full dataset inference — annotated images + COCO predictions JSON.
## Uses VISUALISE_THRESHOLD from the cell above for the saved annotated images.
## Set SAVE_THRESHOLD here independently if you prefer.
SAVE_THRESHOLD = 0.45

import os, json, shutil
from PIL import Image
from tqdm import tqdm
from datetime import datetime

ANNOTATED_IMAGES_DIR  = os.path.join(OUTPUT_DIR, 'annotated_images')
COCO_ANNOTATIONS_PATH = os.path.join(OUTPUT_DIR, 'predictions_coco.json')
os.makedirs(ANNOTATED_IMAGES_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

image_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
test_images = sorted([f for f in os.listdir(TEST_IMAGES_DIR)
                      if f.lower().endswith(image_extensions)])
print(f'Found {len(test_images)} images. Saving detections at conf >= {SAVE_THRESHOLD}')

coco_data = {
    'info': {'description': 'RF-DETR ONNX + SAHI', 'date_created': datetime.now().isoformat(),
             'model': 'RFDETRLarge ONNX', 'threshold': SAVE_THRESHOLD, 'slice_size': SLICE_SIZE},
    'images': [], 'annotations': [],
    'categories': [{'id': i, 'name': n, 'supercategory': 'none'}
                   for i, n in enumerate(CLASS_NAMES)],
}
annotation_id = image_id = 0

color = sv.ColorPalette.from_hex([
    '#ffff00', '#ff9b00', '#ff66ff', '#3399ff', '#ff66b2', '#ff8080',
    '#b266ff', '#9999ff', '#66ffff', '#33ff99', '#66ff66', '#99ff00'
])

for img_filename in tqdm(test_images, desc='Saving annotated images'):
    img_path  = os.path.join(TEST_IMAGES_DIR, img_filename)
    image_pil = Image.open(img_path).convert('RGB')
    width, height = image_pil.size

    # Re-use raw predictions collected in Cell 13, filtered at SAVE_THRESHOLD
    img_idx = test_images.index(img_filename)
    det     = all_raw_preds[img_idx]
    if det.confidence is not None and len(det) > 0:
        det = det[det.confidence >= SAVE_THRESHOLD]
    if det.class_id is not None:
        det.class_id = np.clip(det.class_id, 0, len(CLASS_NAMES) - 1)

    coco_data['images'].append({'id': image_id, 'file_name': img_filename,
                                 'width': width, 'height': height})
    if len(det) > 0:
        for i in range(len(det)):
            x1, y1, x2, y2 = det.xyxy[i]
            bw, bh = x2 - x1, y2 - y1
            coco_data['annotations'].append({
                'id': annotation_id, 'image_id': image_id,
                'category_id': int(det.class_id[i]) if det.class_id is not None else 0,
                'bbox': [float(x1), float(y1), float(bw), float(bh)],
                'area': float(bw * bh),
                'score': float(det.confidence[i]) if det.confidence is not None else 1.0,
                'iscrowd': 0,
            })
            annotation_id += 1

    text_scale = sv.calculate_optimal_text_scale(resolution_wh=(width, height))
    thickness  = sv.calculate_optimal_line_thickness(resolution_wh=(width, height))
    bbox_annotator  = sv.BoxAnnotator(color=color, thickness=thickness)
    label_annotator = sv.LabelAnnotator(color=color, text_color=sv.Color.BLACK,
                                         text_scale=text_scale, smart_position=True)
    ann_img = image_pil.copy()
    if len(det) > 0:
        labels  = [f'object {c:.2f}' if c else 'object'
                   for c in (det.confidence if det.confidence is not None
                             else [None] * len(det))]
        ann_img = bbox_annotator.annotate(ann_img, det)
        ann_img = label_annotator.annotate(ann_img, det, labels)
    ann_img.save(os.path.join(ANNOTATED_IMAGES_DIR, img_filename))
    image_id += 1

with open(COCO_ANNOTATIONS_PATH, 'w') as f:
    json.dump(coco_data, f, indent=2)

print(f'\n✅ Done!')
print(f'   - Processed {len(test_images)} images at conf >= {SAVE_THRESHOLD}')
print(f'   - Found {annotation_id} total detections')
print(f'   - Annotated images: {ANNOTATED_IMAGES_DIR}')
print(f'   - COCO JSON:        {COCO_ANNOTATIONS_PATH}')

try:
    shutil.copytree(OUTPUT_DIR, DRIVE_OUTPUT, dirs_exist_ok=True)
    print(f'   - Copied to Drive: {DRIVE_OUTPUT}')
except Exception as e:
    print(f'   - Could not copy to Drive: {e}')

In [ ]:
import json
from collections import Counter

with open(COCO_ANNOTATIONS_PATH) as f:
    coco_out = json.load(f)

total    = len(coco_out['annotations'])
n_images = len(coco_out['images'])
print(f'Total detections:          {total}')
print(f'Images:                    {n_images}')
print(f'Mean detections per image: {total/n_images:.1f}')

counts = Counter(a['image_id'] for a in coco_out['annotations'])
print('\nTop 5 images by detection count:')
for img_id, count in counts.most_common(5):
    fname = next(i['file_name'] for i in coco_out['images'] if i['id'] == img_id)
    print(f'  {fname}: {count} detections')

In [ ]:
from IPython.display import display
from PIL import Image

sample = os.path.join(ANNOTATED_IMAGES_DIR, test_images[0])
if os.path.exists(sample):
    display(Image.open(sample))
    print(f'\n📸 Preview: {test_images[0]}')

In [ ]:
# Check actual object sizes in your test set
import numpy as np
widths = []
heights = []
for path, img, ann in ds:
    for box in ann.xyxy:
        widths.append(box[2] - box[0])
        heights.append(box[3] - box[1])
print(f'Object width  — mean: {np.mean(widths):.1f}px  median: {np.median(widths):.1f}px  min: {np.min(widths):.1f}px')
print(f'Object height — mean: {np.mean(heights):.1f}px  median: {np.median(heights):.1f}px  min: {np.min(heights):.1f}px')